# 04 - Morphometric Characters

Following source `generate_chars.py` — computes ~70 primary characters across 4 element types:
1. **Street chars** — network metrics, street profile, reached cells/area
2. **Enclosure chars** — shape, neighbours, building coverage
3. **Building chars** — shape, adjacency, alignment, connected structures
4. **Tessellation chars** — shape, neighbours, street alignment, reached enclosures

Canadian additions: `HEIGHT`, `volume`, `floor_count` (from OBPL-CA-v3)
CHM additions: `chm_mean`, `chm_std`, `chm_max`, `chm_cover` (LiDAR canopy stats, passed through from buildings source)

In [1]:
import os
if os.path.basename(os.getcwd()).startswith("notebooks"):
    os.chdir(os.path.dirname(os.getcwd()))
print("Working dir:", os.getcwd())

import sys
sys.path.insert(0, "scripts")
from pipeline_config import load_city_config
load_city_config()

Working dir: C:\urban-form\urban_taxonomy
[pipeline_config] City config: config\canada.env


WindowsPath('config/canada.env')

In [2]:
import gc
import geopandas as gpd
import pandas as pd
import numpy as np
import momepy as mm
from libpysal.graph import Graph, read_parquet
from shapely import unary_union
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

In [3]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# Paths derive from OUTPUT_ROOT (set in the city config, see notebook 01 /
# scripts/pipeline_config.py) so they follow the rest of the pipeline per city.
# ──────────────────────────────────────────────────────────────────────────────

OUTPUT_ROOT = os.environ.get("OUTPUT_ROOT", "outputs")
PREPROCESSED_DIR = Path(OUTPUT_ROOT) / "01_preprocessed"
TESSELLATION_DIR = Path(OUTPUT_ROOT) / "02_tessellation"
OUT       = Path(OUTPUT_ROOT) / "04_chars"
GRAPH_DIR = Path(OUTPUT_ROOT) / "03_graphs"
OUT.mkdir(parents=True, exist_ok=True)

# CHM column names — must match what was set in notebook 01
CHM_COLS = [c.strip() for c in os.environ.get("CHM_COLS", "chm_mean,chm_std,chm_max,chm_cover").split(",") if c.strip()]

# Informational only: notebook 01 drops these columns from buildings.gpkg
# entirely when INCLUDE_CHM is False, so the `if col in buildings.columns`
# checks below already skip them — nothing else to toggle here.
INCLUDE_CHM = os.environ.get("INCLUDE_CHM", "1").strip().lower() not in ("0", "false", "no")

## Load all data

In [4]:
buildings    = gpd.read_file(PREPROCESSED_DIR / "buildings.gpkg")
streets      = gpd.read_file(PREPROCESSED_DIR / "roads.gpkg")
tessellation = gpd.read_parquet(TESSELLATION_DIR / "tessellation.parquet")
enclosures   = gpd.read_parquet(TESSELLATION_DIR / "enclosures.parquet")

print(f"Buildings: {len(buildings):,}, Streets: {len(streets):,}")
print(f"Tessellation: {len(tessellation):,}, Enclosures: {len(enclosures):,}")

Buildings: 84,142, Streets: 12,534
Tessellation: 93,234, Enclosures: 24,052


## 1. Street characters

Source `process_street_chars()` computes:
- Node degree, subgraph metrics (meshedness, proportions, closeness, node density)
- cds_length, clustering, mean_node_dist
- Edge: length, orientation, linearity, mean segment length (k=3)
- Street profile (width, openness, width_deviation)
- Buildings per metre, building area variability
- Tessellation reached cells/area

In [5]:
print("Processing street characters...")

graph = mm.gdf_to_nx(streets, preserve_index=True)
graph = mm.node_degree(graph)

# Subgraph metrics (radius=5 hops)
graph = mm.subgraph(
    graph, radius=5,
    meshedness=True, cds_length=False, mode="sum",
    degree="degree", length="mm_len", mean_node_degree=False,
    proportion={0: True, 3: True, 4: True},
    cyclomatic=False, edge_node_ratio=False, gamma=False,
    local_closeness=True, closeness_weight="mm_len",
    node_density=True, verbose=False,
)

graph = mm.cds_length(graph, radius=3, name="ldsCDL", verbose=False)
graph = mm.clustering(graph, name="xcnSCl")
graph = mm.mean_node_dist(graph, name="mtdMDi", verbose=False)

nodes, edges = mm.nx_to_gdf(graph, spatial_weights=False)
print(f"  Nodes: {len(nodes):,}, Edges: {len(edges):,}")

Processing street characters...


  Nodes: 14,114, Edges: 12,534


In [6]:
# Edge-level characters
edges["sdsLen"] = edges.geometry.length
edges["sssLin"] = mm.linearity(edges)

# Mean segment length within 3 hops
str_q1 = read_parquet(GRAPH_DIR / "street_graph.parquet")

def mean_edge_length(partial_focals, partial_higher, y):
    return partial_higher.describe(
        y.loc[partial_higher.unique_ids], statistics=["mean"]
    )["mean"]

# Simplified version: use 1st-order neighbour mean instead of full partial_apply
edges["ldsMSL"] = str_q1.describe(edges.geometry.length, statistics=["mean"])["mean"]

print(f"  Edge chars computed")

  Edge chars computed


In [7]:
# Street-tessellation interactions
tess_nid = mm.get_nearest_street(tessellation, edges)

edges["sdsAre"] = mm.describe_agg(
    tessellation.geometry.area, tess_nid, statistics=["count", "sum"]
)["sum"]

# Street-building interactions
blg_nid = mm.get_nearest_street(buildings, edges)
edges["sisBpM"] = blg_nid.value_counts() / edges.geometry.length
edges["misBAD"] = buildings.area.groupby(blg_nid).std().fillna(0)

# Street profile
profile = mm.street_profile(edges, buildings, height=None, distance=3)
edges["sdsSPW"] = profile["width"]
edges["sdsSPO"] = profile["openness"]
edges["sdsSWD"] = profile["width_deviation"]

# Node-building interactions
bnode_id = mm.get_nearest_node(buildings, nodes, edges, blg_nid)
nodes["midBAD"] = buildings.area.groupby(bnode_id).std().fillna(0)

# Node-tessellation interactions
nodes_graph = read_parquet(GRAPH_DIR / "nodes_graph.parquet")
edges["nID"] = edges.index.values
tessellation["nID"] = tess_nid
tessellation["nodeID"] = mm.get_nearest_node(tessellation, nodes, edges, tessellation["nID"])

nodes["sddAre"] = mm.describe_agg(tessellation.geometry.area, tessellation["nodeID"])["sum"]
res = mm.describe_reached_agg(tessellation.geometry.area, tessellation["nodeID"], graph=nodes_graph)
nodes["midRea"] = res["count"]
nodes["midAre"] = res["sum"]

# Rename source column names
nodes.rename(columns={
    "degree": "mtdDeg", "meshedness": "lcdMes",
    "local_closeness": "lcnClo",
    "proportion_3": "linP3W", "proportion_4": "linP4W", "proportion_0": "linPDE",
    "node_density": "lddNDe", "node_density_weighted": "linWID",
}, inplace=True)

print("Street + node chars complete")

Street + node chars complete


In [8]:
nodes.to_parquet(OUT / "nodes_chars.parquet")
edges.to_parquet(OUT / "streets_chars.parquet")
print(f"  Saved nodes_chars ({len(nodes):,}) and streets_chars ({len(edges):,})")

  Saved nodes_chars (14,114) and streets_chars (12,534)


## 2. Enclosure characters

In [9]:
print("Processing enclosure characters...")
enclosures["ldkAre"] = enclosures.geometry.area
enclosures["ldkPer"] = enclosures.geometry.length
enclosures["lskCCo"] = mm.circular_compactness(enclosures)
enclosures["lskERI"] = mm.equivalent_rectangular_index(enclosures)
enclosures["lskCWA"] = mm.compactness_weighted_axis(enclosures)
enclosures["ltkOri"] = mm.orientation(enclosures)

# Enclosure neighbour weights
encl_q1 = read_parquet(GRAPH_DIR / "enclosure_graph.parquet")
enclosures["ltkWNB"] = mm.neighbors(enclosures, encl_q1, weighted=True)

# Tessellation cells per enclosure area
encl_counts = tessellation.groupby("enclosure_index").count()
enclosures["likWCe"] = encl_counts["geometry"] / enclosures.geometry.area

# Building coverage in enclosures
beid = buildings.merge(
    tessellation["enclosure_index"], left_index=True, right_index=True
)["enclosure_index"]
res = mm.describe_agg(buildings.geometry.area, beid, statistics=["count", "sum"])
enclosures["likWBB"] = res["sum"] / enclosures.geometry.area

enclosures.to_parquet(OUT / "enclosures_chars.parquet")
print(f"  Enclosure chars saved ({len(enclosures):,})")
del encl_q1; gc.collect()

Processing enclosure characters...


  Enclosure chars saved (24,052)


57098

## 3. Building characters

Shape, adjacency, alignment, connected structures (9 features from union of touching buildings)

In [10]:
print("Processing building characters...")

# Shape
buildings["ssbCCo"] = mm.circular_compactness(buildings)
buildings["ssbCor"] = mm.corners(buildings, eps=15)
buildings.loc[buildings["ssbCCo"] >= 0.95, "ssbCor"] = 0
buildings["ssbSqu"] = mm.squareness(buildings, eps=15)
buildings.loc[buildings["ssbCCo"] >= 0.95, "ssbSqu"] = 90
cencon = mm.centroid_corner_distance(buildings, eps=15)
buildings["ssbCCM"] = cencon["mean"]
buildings["ssbCCD"] = cencon["std"]

buildings["sdbAre"] = buildings.geometry.area
buildings["sdbPer"] = buildings.geometry.length
buildings["sdbCoA"] = mm.courtyard_area(buildings.geometry)
buildings["ssbERI"] = mm.equivalent_rectangular_index(buildings.geometry)
buildings["ssbElo"] = mm.elongation(buildings.geometry)
buildings["stbOri"] = mm.orientation(buildings)

# Shared walls ratio
try:
    buildings["mtbSWR"] = (
        mm.shared_walls(buildings, strict=False, tolerance=0.15)
        / buildings.geometry.length
    )
except Exception:
    buildings["mtbSWR"] = (
        mm.shared_walls(buildings.set_precision(1e-6), strict=False, tolerance=0.15)
        / buildings.geometry.length
    )

print(f"  Shape chars computed")

Processing building characters...


  Shape chars computed


In [11]:
# Connected structure + adjacency chars
buildings_q1 = read_parquet(GRAPH_DIR / "building_graph.parquet")
buildings["libNCo"] = mm.courtyards(buildings, buildings_q1, buffer=0.25)
buildings["ldbPWL"] = mm.perimeter_wall(buildings, buildings_q1, buffer=0.25)

# Connected building structures
connected = buildings.geometry.groupby(
    buildings_q1.component_labels
).apply(lambda x: unary_union(x.values))
connected = connected.set_crs(buildings.crs).buffer(0.1).normalize().make_valid()
comps = buildings_q1.component_labels

buildings.loc[comps.index, "mibCou"] = comps.value_counts().loc[comps.values].values
buildings.loc[comps.index, "mibLen"] = connected.length.loc[comps.values].values
buildings.loc[comps.index, "mibAre"] = connected.area.loc[comps.values].values
buildings.loc[comps.index, "mibElo"] = mm.elongation(connected).loc[comps.values].values
buildings.loc[comps.index, "mibERI"] = mm.equivalent_rectangular_index(connected).loc[comps.values].values
buildings.loc[comps.index, "mibCCo"] = mm.circular_compactness(connected).loc[comps.values].values
buildings.loc[comps.index, "mibLAL"] = mm.longest_axis_length(connected).loc[comps.values].values
buildings.loc[comps.index, "mibFR"]  = mm.facade_ratio(connected).loc[comps.values].values
buildings.loc[comps.index, "mibSCo"] = mm.square_compactness(connected).loc[comps.values].values

del connected
print(f"  Connected structure chars computed")

  Connected structure chars computed


In [12]:
# Tessellation-based building context
queen_1 = read_parquet(GRAPH_DIR / "tessellation_graph.parquet")
bgraph = queen_1.subgraph(buildings_q1.unique_ids)

buildings["ltcBuA"] = mm.building_adjacency(buildings_q1, bgraph)
buildings["mtbAli"] = mm.alignment(buildings["stbOri"], bgraph)
buildings["mtbNDi"] = mm.neighbor_distance(buildings, bgraph)

# Mean interbuilding distance (simplified: use 1st order only)
buildings["ltbIBD"] = mm.mean_interbuilding_distance(buildings, buildings_q1, bgraph)

# Cell alignment
tessellation["stcOri"] = mm.orientation(tessellation)
buildings["stbCeA"] = mm.cell_alignment(
    buildings["stbOri"], tessellation[tessellation.index >= 0]["stcOri"]
)

# Street alignment
blg_nid = mm.get_nearest_street(buildings, edges)
street_orientation = mm.orientation(streets)
buildings["nID"] = blg_nid
edges["nID"] = edges.index.values
buildings["stbSAl"] = mm.street_alignment(
    buildings["stbOri"][~blg_nid.isna()],
    street_orientation,
    blg_nid[~blg_nid.isna()],
)
buildings["nodeID"] = mm.get_nearest_node(buildings, nodes, edges, buildings["nID"])

# Canadian additions: volume, floor count
buildings["volume"] = buildings["sdbAre"] * buildings["HEIGHT"]
buildings["floor_count"] = (buildings["HEIGHT"] / 3.0).clip(lower=1).round()

# CHM canopy stats are already present from the source GPKG; fill any remaining
# NaNs with 0 in case a building was dropped/re-indexed during cleaning steps.
for col in CHM_COLS:
    if col in buildings.columns:
        buildings[col] = buildings[col].fillna(0).clip(lower=0)

buildings.to_parquet(OUT / "buildings_chars.parquet")
print(f"  Building chars saved ({len(buildings):,}, {len([c for c in buildings.columns if c not in ['geometry']])} columns)")

del queen_1, bgraph, buildings_q1; gc.collect()

  Building chars saved (84,142, 49 columns)


0

## 4. Tessellation characters

In [13]:
print("Processing tessellation characters...")

tessellation["stcOri"] = mm.orientation(tessellation)
tessellation["sdcLAL"] = mm.longest_axis_length(tessellation)
tessellation["sdcAre"] = tessellation.geometry.area
tessellation["sscCCo"] = mm.circular_compactness(tessellation)
tessellation["sscERI"] = mm.equivalent_rectangular_index(tessellation.geometry)

queen_1 = read_parquet(GRAPH_DIR / "tessellation_graph.parquet")
tessellation["mtcWNe"] = mm.neighbors(tessellation, queen_1, weighted=True)
tessellation["mdcAre"] = queen_1.describe(
    tessellation.geometry.area, statistics=["sum"]
)["sum"]

# Covered area ratio
tessellation["sicCAR"] = buildings.geometry.area / tessellation.geometry.area

# Building area deviation in tessellation neighbourhood
tessellation["barea"] = buildings.area
tessellation["barea"] = tessellation["barea"].fillna(0)
res = queen_1.describe(tessellation["barea"])
tessellation["micBAD"] = res["std"]

# Street alignment of tessellation
tess_nid = mm.get_nearest_street(tessellation, edges)
street_orientation = mm.orientation(edges)
tessellation["stcSAl"] = mm.street_alignment(
    tessellation["stcOri"], street_orientation, tess_nid.astype(int),
)

edges["nID"] = edges.index.values
tessellation["nID"] = tess_nid
tessellation["nodeID"] = mm.get_nearest_node(
    tessellation, nodes, edges, tessellation["nID"]
)

tessellation.to_parquet(OUT / "tessellations_chars.parquet")
print(f"  Tessellation chars saved ({len(tessellation):,})")

del queen_1; gc.collect()
print("\nAll character files saved to", OUT)

Processing tessellation characters...


  Tessellation chars saved (93,234)

All character files saved to outputs_canada\regions\4810\04_chars
